# Advanced YAML with PyYAML — Tutorial-Style Problems and Solutions

This notebook continues the YAML/PyYAML topic, but uses a **guided tutorial style**.

Instead of jumping directly into large exercises, each section follows a pattern similar to a classroom notebook:

1. introduce one idea,
2. inspect a small example,
3. explain what happened,
4. build the idea into a problem,
5. solve the problem in logical steps,
6. finish with observations and extensions.

The focus is not only on writing code that works, but also on understanding **why** it works and how to structure YAML-related code safely.

We will work with:

- YAML scalar resolution
- nested structures
- safe loading
- serialization
- multi-document YAML
- anchors and aliases
- custom tags
- configuration validation
- configuration inheritance
- environment expansion
- error reporting
- dataclasses
- custom dumpers
- configuration version migration
- final production-style configuration loading


## Setup

PyYAML is a third-party library.

If it is not installed in your environment, install it with:

```bash
python -m pip install pyyaml
```

We will use `yaml.safe_load` as our normal parser.

That is an important default because YAML is capable of expressing more than simple strings, numbers, lists, and dictionaries.


In [ ]:
import os
import re
from copy import deepcopy
from dataclasses import dataclass, asdict
from datetime import date
from pathlib import Path
from pprint import pprint
from typing import Any

import yaml

print("PyYAML version:", yaml.__version__)


Before we start the larger problems, let's define one rule that will be used throughout this notebook.

> **Rule:** YAML parsing and application validation are two different jobs.

PyYAML can tell us whether some text is valid YAML.

It cannot automatically know application rules such as:

- a port must be between `1` and `65535`,
- a timeout must be positive,
- two services cannot use the same name,
- production cannot run with debug mode enabled.

Those rules belong in our own validation layer.


# Problem 1 — Understanding YAML Scalar Resolution

Let's begin with something that looks simple but causes many real-world configuration bugs: **automatic scalar conversion**.

YAML does not treat every unquoted value as a string.

For example:

```yaml
enabled: true
count: 10
price: 4.5
release: 2026-08-07
nothing: null
```

PyYAML may convert those values into Python types automatically.


### Step 1 — Parse a small YAML document

We will start by loading several scalar values and inspecting their Python values.


In [ ]:
scalar_yaml = '''
enabled: true
disabled: false
count: 10
ratio: 4.5
nothing: null
release_date: 2026-08-07
name: example
'''

scalar_data = yaml.safe_load(scalar_yaml)

pprint(scalar_data)


### Step 2 — Inspect the resulting types

Printing the dictionary is useful, but it does not always make the types obvious.

Let's inspect each value individually.


In [ ]:
for key, value in scalar_data.items():
    print(f"{key:15} -> {value!r:20} type={type(value).__name__}")


Notice that `release_date` is not merely text.

PyYAML recognized the ISO-looking date and created a `datetime.date` object.

This can be convenient, but it can also be surprising.


### Step 3 — Force ambiguous values to remain strings

Suppose an application needs a release identifier such as `2026-08-07` to remain textual.

We can quote it in YAML.


In [ ]:
quoted_yaml = '''
release_date: "2026-08-07"
count: "10"
enabled: "true"
'''

quoted_data = yaml.safe_load(quoted_yaml)

for key, value in quoted_data.items():
    print(f"{key:15} -> {value!r:20} type={type(value).__name__}")


## Your Problem

Write YAML containing all of the following:

- a Boolean,
- an integer,
- a floating-point number,
- a null value,
- a date,
- a date-looking string,
- a number-looking string.

Then parse it and verify each Python type with assertions.


### Solution

The important part is not merely producing the YAML.

We also verify the behavior explicitly so that a future change in configuration does not silently change an application's expectations.


In [ ]:
problem_1_yaml = '''
active: true
retries: 5
threshold: 0.85
optional_value: null
created_on: 2025-12-31
created_on_text: "2025-12-31"
ticket_number: "00042"
'''

problem_1 = yaml.safe_load(problem_1_yaml)

assert isinstance(problem_1["active"], bool)
assert isinstance(problem_1["retries"], int)
assert isinstance(problem_1["threshold"], float)
assert problem_1["optional_value"] is None
assert isinstance(problem_1["created_on"], date)
assert isinstance(problem_1["created_on_text"], str)
assert isinstance(problem_1["ticket_number"], str)

pprint(problem_1)


### Observation

When the exact textual representation matters, quote the value.

This is especially important for:

- identifiers with leading zeros,
- date-like IDs,
- values that look Boolean,
- values consumed by systems with stricter schemas.


# Problem 2 — Navigating a Realistic Nested YAML Document

Simple YAML examples are useful, but production configuration usually contains several levels of mappings and sequences.

Consider an application containing:

- metadata,
- database configuration,
- background jobs,
- feature flags.


In [ ]:
application_yaml = '''
application:
  name: billing-service
  version: "3.2"

database:
  primary:
    host: db-primary.internal
    port: 5432
  replicas:
    - host: db-replica-1.internal
      port: 5432
    - host: db-replica-2.internal
      port: 5432

workers:
  - name: invoices
    concurrency: 6
    enabled: true
  - name: reminders
    concurrency: 2
    enabled: false
  - name: reports
    concurrency: 3
    enabled: true

features:
  instant_refund: true
  legacy_export: false
'''

application = yaml.safe_load(application_yaml)


### Step 1 — Access a deeply nested value

The primary database host is found by following mapping keys.


In [ ]:
primary_host = application["database"]["primary"]["host"]
print(primary_host)


### Step 2 — Work with a YAML sequence

`workers` became a Python list.

Each element of that list is itself a dictionary.


In [ ]:
for worker in application["workers"]:
    print(worker["name"], worker["concurrency"], worker["enabled"])


### Step 3 — Derive useful data

Configuration is rarely loaded just to be printed.

Usually we transform it into structures that are convenient for the rest of the application.


In [ ]:
enabled_workers = [
    worker
    for worker in application["workers"]
    if worker["enabled"]
]

worker_concurrency = {
    worker["name"]: worker["concurrency"]
    for worker in enabled_workers
}

pprint(worker_concurrency)


## Your Problem

Using the parsed configuration:

1. collect all replica host names,
2. calculate total concurrency of enabled workers,
3. produce a list of disabled feature names,
4. create a summary dictionary containing those three results.


### Solution — Part 1: Replica hosts


In [ ]:
replica_hosts = [
    replica["host"]
    for replica in application["database"]["replicas"]
]

replica_hosts


### Solution — Part 2: Enabled worker concurrency


In [ ]:
total_enabled_concurrency = sum(
    worker["concurrency"]
    for worker in application["workers"]
    if worker["enabled"]
)

total_enabled_concurrency


### Solution — Part 3: Disabled feature names


In [ ]:
disabled_features = [
    name
    for name, enabled in application["features"].items()
    if not enabled
]

disabled_features


### Solution — Part 4: Build the final summary


In [ ]:
summary = {
    "replica_hosts": replica_hosts,
    "enabled_worker_concurrency": total_enabled_concurrency,
    "disabled_features": disabled_features,
}

pprint(summary)

assert summary["replica_hosts"] == [
    "db-replica-1.internal",
    "db-replica-2.internal",
]
assert summary["enabled_worker_concurrency"] == 9
assert summary["disabled_features"] == ["legacy_export"]


# Problem 3 — Controlling YAML Serialization

Parsing converts YAML into Python objects.

Serialization performs the reverse transformation.

For configuration files, we usually want generated YAML to be:

- readable,
- stable,
- easy to diff,
- free from Python-specific object tags.


### Step 1 — Start with ordinary Python data


In [ ]:
service_data = {
    "service": "search",
    "enabled": True,
    "replicas": 3,
    "regions": [
        "eu-west-1",
        "eu-central-1",
    ],
    "limits": {
        "requests_per_second": 150,
        "burst": 300,
    },
}


### Step 2 — Use `safe_dump`

We will preserve insertion order with `sort_keys=False`.


In [ ]:
service_yaml = yaml.safe_dump(
    service_data,
    sort_keys=False,
    default_flow_style=False,
)

print(service_yaml)


### Step 3 — Verify round-trip behavior

A good serialization exercise should not stop at visually inspecting the text.

We can load it back and compare the resulting data.


In [ ]:
restored_service_data = yaml.safe_load(service_yaml)

assert restored_service_data == service_data

print("Round-trip successful.")


## Your Problem

Create a Python dictionary representing an API gateway with:

- a Unicode description,
- a list of allowed methods,
- nested timeout settings,
- a Boolean flag.

Serialize it with readable block style, preserve insertion order, and verify that loading the generated YAML recreates the original dictionary.


### Solution


In [ ]:
gateway = {
    "name": "edge-gateway",
    "description": "Маршрутизатор за API заявки",
    "methods": ["GET", "POST", "DELETE"],
    "timeouts": {
        "connect": 2.0,
        "read": 15.0,
    },
    "compression": True,
}

gateway_yaml = yaml.safe_dump(
    gateway,
    sort_keys=False,
    default_flow_style=False,
    allow_unicode=True,
)

print(gateway_yaml)

assert yaml.safe_load(gateway_yaml) == gateway


# Problem 4 — Multi-Document YAML Streams

A YAML stream may contain several documents separated by `---`.

This is common in infrastructure configuration and deployment manifests.

For example, one file may contain several logical resources.


In [ ]:
documents_yaml = '''
---
kind: Queue
name: emails
max_size: 1000
enabled: true

---
kind: Queue
name: billing
max_size: 500
enabled: false

---
kind: Queue
name: reports
max_size: 250
enabled: true
'''


### Step 1 — Why `safe_load` is not enough

`safe_load` expects one YAML document.

For a stream containing several documents we use `safe_load_all`.


In [ ]:
documents = list(yaml.safe_load_all(documents_yaml))

pprint(documents)


### Step 2 — Process every document

Once loaded, the YAML stream behaves like a normal Python list of dictionaries.


In [ ]:
enabled_documents = [
    doc
    for doc in documents
    if doc["enabled"]
]

pprint(enabled_documents)


## Your Problem

From the multi-document stream:

1. verify all documents have `kind == "Queue"`,
2. collect the names of enabled queues,
3. calculate the total `max_size` of enabled queues,
4. serialize only the enabled queues into another YAML stream.


### Solution


In [ ]:
for doc in documents:
    assert doc["kind"] == "Queue"

enabled_queue_names = [
    doc["name"]
    for doc in documents
    if doc["enabled"]
]

enabled_capacity = sum(
    doc["max_size"]
    for doc in documents
    if doc["enabled"]
)

enabled_stream = yaml.safe_dump_all(
    enabled_documents,
    explicit_start=True,
    sort_keys=False,
)

print("Enabled queues:", enabled_queue_names)
print("Enabled capacity:", enabled_capacity)
print()
print(enabled_stream)

assert enabled_queue_names == ["emails", "reports"]
assert enabled_capacity == 1250


# Problem 5 — Anchors, Aliases, and Shared Configuration

YAML anchors let us define reusable content.

Aliases refer back to that content.

Merge keys can then reuse a mapping while still allowing overrides.


Consider this deployment configuration.


In [ ]:
anchor_yaml = '''
defaults: &service_defaults
  image: mycompany/app:3.0
  replicas: 2
  restart: always
  resources:
    cpu: "500m"
    memory: "256Mi"

staging:
  <<: *service_defaults
  environment: staging

production:
  <<: *service_defaults
  environment: production
  replicas: 8
  resources:
    cpu: "2"
    memory: "1Gi"
'''

anchor_data = yaml.safe_load(anchor_yaml)

pprint(anchor_data)


### Step 1 — Inspect inheritance

The staging configuration inherited values from the anchor.

The production configuration also inherited them, but explicitly replaced some values.


In [ ]:
print("staging replicas:", anchor_data["staging"]["replicas"])
print("production replicas:", anchor_data["production"]["replicas"])
print("staging image:", anchor_data["staging"]["image"])


### Important Detail

YAML's merge key works at the mapping level.

If a nested mapping is replaced, the replacement is not automatically a deep merge.

In the example above, the entire `resources` mapping in `production` is explicitly supplied.


## Your Problem

Extend the YAML with a `development` environment that:

- inherits the common defaults,
- uses one replica,
- has environment `development`,
- uses only `128Mi` of memory.

Then verify the inherited and overridden values.


### Solution


In [ ]:
anchor_problem_yaml = '''
defaults: &service_defaults
  image: mycompany/app:3.0
  replicas: 2
  restart: always
  resources:
    cpu: "500m"
    memory: "256Mi"

development:
  <<: *service_defaults
  environment: development
  replicas: 1
  resources:
    cpu: "500m"
    memory: "128Mi"
'''

anchor_problem = yaml.safe_load(anchor_problem_yaml)

assert anchor_problem["development"]["image"] == "mycompany/app:3.0"
assert anchor_problem["development"]["restart"] == "always"
assert anchor_problem["development"]["replicas"] == 1
assert anchor_problem["development"]["resources"]["memory"] == "128Mi"

pprint(anchor_problem["development"])


# Problem 6 — Detecting Duplicate Keys

A YAML document can accidentally repeat a key.

For configuration, that is often a mistake:

```yaml
database:
  host: db-one
  host: db-two
```

A parser may keep only one of the values.

For safety-critical configuration, silently accepting the duplicate can be undesirable.


### Step 1 — Create a restricted loader

We will derive from `yaml.SafeLoader`.

That keeps the normal safe-loader behavior while changing only mapping construction.


In [ ]:
class DuplicateCheckingLoader(yaml.SafeLoader):
    pass


### Step 2 — Define a mapping constructor

The constructor checks whether a key has already appeared before adding it to the mapping.


In [ ]:
def construct_mapping_without_duplicates(loader, node, deep=False):
    result = {}

    for key_node, value_node in node.value:
        key = loader.construct_object(key_node, deep=deep)

        if key in result:
            raise yaml.constructor.ConstructorError(
                "while constructing a mapping",
                node.start_mark,
                f"duplicate key found: {key!r}",
                key_node.start_mark,
            )

        value = loader.construct_object(value_node, deep=deep)
        result[key] = value

    return result


### Step 3 — Register the constructor


In [ ]:
DuplicateCheckingLoader.add_constructor(
    yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG,
    construct_mapping_without_duplicates,
)


### Step 4 — Test valid YAML first


In [ ]:
valid_yaml = '''
database:
  host: db.internal
  port: 5432
'''

valid_data = yaml.load(
    valid_yaml,
    Loader=DuplicateCheckingLoader,
)

pprint(valid_data)


### Step 5 — Test a duplicate key


In [ ]:
duplicate_yaml = '''
database:
  host: db-one.internal
  host: db-two.internal
'''

try:
    yaml.load(
        duplicate_yaml,
        Loader=DuplicateCheckingLoader,
    )
except yaml.constructor.ConstructorError as exc:
    print(exc)


## Your Problem

Modify the example to include a duplicate nested key under `logging`.

Verify that the custom loader rejects it before the application sees the configuration.


### Solution


In [ ]:
duplicate_logging_yaml = '''
service:
  name: api

logging:
  level: INFO
  format: text
  level: DEBUG
'''

try:
    yaml.load(
        duplicate_logging_yaml,
        Loader=DuplicateCheckingLoader,
    )
except yaml.constructor.ConstructorError as exc:
    print("Duplicate correctly rejected:")
    print(exc)
else:
    raise AssertionError("The duplicate key should have been rejected")


# Problem 7 — Building Better YAML Error Messages

When a user edits configuration manually, a raw parser traceback is usually not the best error message.

We can catch `yaml.YAMLError` and extract the line and column when PyYAML provides them.


In [ ]:
invalid_indentation_yaml = '''
service:
  name: api
  ports:
    - 8000
    - 8001
   workers: 4
'''


### Step 1 — Write a small parser wrapper


In [ ]:
def parse_yaml_with_message(text: str):
    try:
        return yaml.safe_load(text), None
    except yaml.YAMLError as exc:
        mark = getattr(exc, "problem_mark", None)

        if mark is not None:
            message = (
                f"YAML syntax error at line {mark.line + 1}, "
                f"column {mark.column + 1}: {exc}"
            )
        else:
            message = f"YAML syntax error: {exc}"

        return None, message


### Step 2 — Observe the friendly result


In [ ]:
data, error = parse_yaml_with_message(invalid_indentation_yaml)

print("data:", data)
print()
print(error)


## Your Problem

Create another invalid YAML example involving a malformed sequence.

Use the helper and assert that:

- no data is returned,
- an error message is returned.


### Solution


In [ ]:
malformed_sequence_yaml = '''
servers:
  - name: first
    port: 8000
  - name: second
    port: 8001
  - broken
    value: 123
'''

data, error = parse_yaml_with_message(malformed_sequence_yaml)

assert data is None
assert isinstance(error, str)

print(error)


# Problem 8 — Validation After Parsing

YAML can be syntactically valid but still be unusable for our application.

Consider:

```yaml
name: api
port: 100000
workers: zero
```

This is valid YAML.

It is not a valid service configuration.


### Step 1 — Define the application contract

We want:

- `name`: required, non-empty string,
- `port`: integer from 1 to 65535,
- `workers`: positive integer,
- `debug`: optional Boolean, default `False`.


### Step 2 — Validate one field at a time

We will also explicitly reject booleans where integers are expected because in Python:

```python
isinstance(True, int)
```

is `True`.


In [ ]:
def validate_service(raw: Any) -> dict:
    if not isinstance(raw, dict):
        raise ValueError("service configuration must be a mapping")

    name = raw.get("name")
    port = raw.get("port")
    workers = raw.get("workers", 1)
    debug = raw.get("debug", False)

    if not isinstance(name, str) or not name.strip():
        raise ValueError("'name' must be a non-empty string")

    if isinstance(port, bool) or not isinstance(port, int):
        raise ValueError("'port' must be an integer")

    if not 1 <= port <= 65535:
        raise ValueError("'port' must be between 1 and 65535")

    if isinstance(workers, bool) or not isinstance(workers, int):
        raise ValueError("'workers' must be an integer")

    if workers <= 0:
        raise ValueError("'workers' must be positive")

    if not isinstance(debug, bool):
        raise ValueError("'debug' must be a boolean")

    return {
        "name": name.strip(),
        "port": port,
        "workers": workers,
        "debug": debug,
    }


### Step 3 — Validate a correct document


In [ ]:
service_yaml = '''
name: orders-api
port: 8080
workers: 6
'''

validated_service = validate_service(
    yaml.safe_load(service_yaml)
)

pprint(validated_service)


### Step 4 — Validate several incorrect documents

Testing failure cases is as important as testing the happy path.


In [ ]:
invalid_service_examples = {
    "empty name": '''
name: ""
port: 8080
''',
    "bad port": '''
name: api
port: 99999
''',
    "boolean port": '''
name: api
port: true
''',
    "zero workers": '''
name: api
port: 8080
workers: 0
''',
}

for label, text in invalid_service_examples.items():
    try:
        validate_service(yaml.safe_load(text))
    except ValueError as exc:
        print(f"{label:15} -> {exc}")


## Your Problem

Add a new optional field called `timeout_seconds`.

Rules:

- default is `30`,
- it must be numeric,
- it must be greater than `0`,
- booleans are not accepted.

Write a second validation function that includes this new field.


### Solution


In [ ]:
def validate_service_with_timeout(raw: Any) -> dict:
    result = validate_service(raw)

    timeout = raw.get("timeout_seconds", 30)

    if isinstance(timeout, bool) or not isinstance(timeout, (int, float)):
        raise ValueError("'timeout_seconds' must be numeric")

    if timeout <= 0:
        raise ValueError("'timeout_seconds' must be greater than zero")

    result["timeout_seconds"] = float(timeout)

    return result


service_with_timeout = yaml.safe_load('''
name: reports
port: 9000
workers: 3
timeout_seconds: 12.5
''')

validated = validate_service_with_timeout(service_with_timeout)

pprint(validated)

assert validated["timeout_seconds"] == 12.5


# Problem 9 — Converting Validated YAML into Dataclasses

A common design is to keep YAML-specific work separate from domain objects.

The YAML parser produces ordinary Python data.

Validation normalizes that data.

Only then do we construct our application's objects.


### Step 1 — Define the domain model


In [ ]:
@dataclass(frozen=True)
class ServiceConfig:
    name: str
    port: int
    workers: int = 1
    debug: bool = False
    timeout_seconds: float = 30.0


### Step 2 — Convert validated data into the dataclass


In [ ]:
service_config = ServiceConfig(**validated)

print(service_config)


### Step 3 — Serialize safely

Rather than asking YAML to serialize an arbitrary Python object directly, convert the dataclass to plain data first.


In [ ]:
plain_service_config = asdict(service_config)

print(
    yaml.safe_dump(
        plain_service_config,
        sort_keys=False,
    )
)


## Your Problem

Create two service dataclass instances, place them in a dictionary, convert the entire structure to plain data, serialize it, load it again, and verify the names.


### Solution


In [ ]:
service_a = ServiceConfig(
    name="api",
    port=8000,
    workers=4,
)

service_b = ServiceConfig(
    name="worker-admin",
    port=8001,
    workers=2,
    timeout_seconds=15.0,
)

services_plain = {
    "primary": asdict(service_a),
    "secondary": asdict(service_b),
}

services_text = yaml.safe_dump(
    services_plain,
    sort_keys=False,
)

print(services_text)

restored_services = yaml.safe_load(services_text)

assert restored_services["primary"]["name"] == "api"
assert restored_services["secondary"]["name"] == "worker-admin"


# Problem 10 — Configuration Inheritance with Deep Merge

Applications often have several layers:

```text
built-in defaults
    ↓
base configuration
    ↓
environment configuration
    ↓
runtime overrides
```

A shallow dictionary update is often not enough.


Consider these two configurations.


In [ ]:
base_config_yaml = '''
app:
  name: catalog
  workers: 2

logging:
  level: INFO
  json: false

database:
  host: localhost
  port: 5432
  pool:
    min: 2
    max: 10
'''

production_config_yaml = '''
app:
  workers: 8

logging:
  level: WARNING
  json: true

database:
  host: db.production.internal
  pool:
    max: 50
'''


### Step 1 — Why `dict.update` is insufficient

A normal `update` would replace the entire nested `database` mapping.

We want to preserve unspecified nested values such as:

- database port,
- pool minimum.


### Step 2 — Implement recursive merge


In [ ]:
def deep_merge(base: dict, override: dict) -> dict:
    result = deepcopy(base)

    for key, value in override.items():
        if (
            key in result
            and isinstance(result[key], dict)
            and isinstance(value, dict)
        ):
            result[key] = deep_merge(result[key], value)
        else:
            result[key] = deepcopy(value)

    return result


### Step 3 — Merge the two configurations


In [ ]:
base_config = yaml.safe_load(base_config_yaml)
production_config = yaml.safe_load(production_config_yaml)

merged_config = deep_merge(base_config, production_config)

pprint(merged_config)


### Step 4 — Verify preserved and overridden values


In [ ]:
assert merged_config["app"]["name"] == "catalog"
assert merged_config["app"]["workers"] == 8

assert merged_config["database"]["host"] == "db.production.internal"
assert merged_config["database"]["port"] == 5432

assert merged_config["database"]["pool"]["min"] == 2
assert merged_config["database"]["pool"]["max"] == 50


## Your Problem

Add a third layer representing emergency runtime overrides:

```yaml
logging:
  level: ERROR

database:
  pool:
    max: 20
```

Merge all three layers and verify that only those values change.


### Solution


In [ ]:
runtime_override_yaml = '''
logging:
  level: ERROR

database:
  pool:
    max: 20
'''

runtime_override = yaml.safe_load(runtime_override_yaml)

final_config = deep_merge(
    merged_config,
    runtime_override,
)

pprint(final_config)

assert final_config["logging"]["level"] == "ERROR"
assert final_config["logging"]["json"] is True
assert final_config["database"]["pool"]["min"] == 2
assert final_config["database"]["pool"]["max"] == 20


# Problem 11 — Environment Variable Expansion

Configuration files frequently contain placeholders rather than secrets or machine-specific values.

For example:

```yaml
database:
  host: "${DB_HOST}"
  password: "${DB_PASSWORD}"
```

We will implement interpolation after parsing.


### Step 1 — Define placeholder syntax

We will support:

```text
${NAME}
${NAME:-default value}
```


In [ ]:
ENV_PATTERN = re.compile(
    r"\$\{([A-Za-z_][A-Za-z0-9_]*)(?::-([^}]*))?\}"
)


### Step 2 — Expand placeholders inside one string


In [ ]:
def expand_env_string(value: str, env: dict[str, str]) -> str:
    def replace(match: re.Match) -> str:
        name = match.group(1)
        default = match.group(2)

        if name in env:
            return env[name]

        if default is not None:
            return default

        raise KeyError(f"missing required environment variable: {name}")

    return ENV_PATTERN.sub(replace, value)


### Step 3 — Apply the operation recursively

Placeholders may appear in deeply nested dictionaries or lists.


In [ ]:
def interpolate_environment(value: Any, env: dict[str, str]) -> Any:
    if isinstance(value, str):
        return expand_env_string(value, env)

    if isinstance(value, list):
        return [
            interpolate_environment(item, env)
            for item in value
        ]

    if isinstance(value, dict):
        return {
            key: interpolate_environment(item, env)
            for key, item in value.items()
        }

    return value


### Step 4 — Try the interpolation


In [ ]:
env_yaml = '''
database:
  host: "${DB_HOST:-localhost}"
  user: "${DB_USER}"
  password: "${DB_PASSWORD}"

jobs:
  - name: daily-report
    token: "${REPORT_TOKEN:-development-token}"
'''

env_config = yaml.safe_load(env_yaml)

expanded_config = interpolate_environment(
    env_config,
    {
        "DB_HOST": "db.internal",
        "DB_USER": "app_user",
        "DB_PASSWORD": "secret-value",
    },
)

pprint(expanded_config)


## Your Problem

Add two nested webhook definitions.

One should require `${PAYMENTS_WEBHOOK}`.

The other should use `${AUDIT_WEBHOOK:-http://localhost:9000/audit}`.

Expand both with an environment dictionary that provides only the payments webhook.


### Solution


In [ ]:
webhook_yaml = '''
webhooks:
  payments:
    url: "${PAYMENTS_WEBHOOK}"
  audit:
    url: "${AUDIT_WEBHOOK:-http://localhost:9000/audit}"
'''

webhook_config = yaml.safe_load(webhook_yaml)

expanded_webhooks = interpolate_environment(
    webhook_config,
    {
        "PAYMENTS_WEBHOOK": "https://hooks.example/payments",
    },
)

pprint(expanded_webhooks)

assert expanded_webhooks["webhooks"]["payments"]["url"] == (
    "https://hooks.example/payments"
)
assert expanded_webhooks["webhooks"]["audit"]["url"] == (
    "http://localhost:9000/audit"
)


# Problem 12 — Creating a Safe Custom YAML Tag

YAML can support custom tags.

Instead of allowing arbitrary Python object construction, we can define a narrow tag with a carefully controlled constructor.

We will create:

```yaml
timeout: !duration 2.5s
```

The result will be a number of seconds.


### Step 1 — Define the accepted duration format

Supported units:

- `ms`
- `s`
- `m`
- `h`


In [ ]:
DURATION_PATTERN = re.compile(
    r"^(\d+(?:\.\d+)?)(ms|s|m|h)$"
)

DURATION_MULTIPLIERS = {
    "ms": 0.001,
    "s": 1.0,
    "m": 60.0,
    "h": 3600.0,
}


### Step 2 — Derive from `SafeLoader`

This is important.

We want to add one known feature without enabling arbitrary Python-specific YAML tags.


In [ ]:
class DurationLoader(yaml.SafeLoader):
    pass


### Step 3 — Write the constructor


In [ ]:
def construct_duration(loader, node):
    text = loader.construct_scalar(node)

    match = DURATION_PATTERN.fullmatch(text)

    if match is None:
        raise yaml.constructor.ConstructorError(
            None,
            None,
            f"invalid duration value: {text!r}",
            node.start_mark,
        )

    number = float(match.group(1))
    unit = match.group(2)

    return number * DURATION_MULTIPLIERS[unit]


### Step 4 — Register the tag


In [ ]:
DurationLoader.add_constructor(
    "!duration",
    construct_duration,
)


### Step 5 — Parse valid values


In [ ]:
duration_yaml = '''
connect_timeout: !duration 250ms
read_timeout: !duration 2.5s
cache_ttl: !duration 10m
job_timeout: !duration 1h
'''

duration_data = yaml.load(
    duration_yaml,
    Loader=DurationLoader,
)

pprint(duration_data)


### Step 6 — Verify invalid values fail


In [ ]:
try:
    yaml.load(
        "timeout: !duration tomorrow",
        Loader=DurationLoader,
    )
except yaml.constructor.ConstructorError as exc:
    print(exc)


## Your Problem

Add support for a new `d` unit representing days.

Then parse:

```yaml
retention: !duration 7d
```

and verify that the result is `604800.0`.


### Solution


In [ ]:
DURATION_PATTERN_WITH_DAYS = re.compile(
    r"^(\d+(?:\.\d+)?)(ms|s|m|h|d)$"
)

DURATION_MULTIPLIERS_WITH_DAYS = {
    **DURATION_MULTIPLIERS,
    "d": 86400.0,
}


class DurationWithDaysLoader(yaml.SafeLoader):
    pass


def construct_duration_with_days(loader, node):
    text = loader.construct_scalar(node)

    match = DURATION_PATTERN_WITH_DAYS.fullmatch(text)

    if match is None:
        raise yaml.constructor.ConstructorError(
            None,
            None,
            f"invalid duration value: {text!r}",
            node.start_mark,
        )

    number = float(match.group(1))
    unit = match.group(2)

    return number * DURATION_MULTIPLIERS_WITH_DAYS[unit]


DurationWithDaysLoader.add_constructor(
    "!duration",
    construct_duration_with_days,
)

retention = yaml.load(
    "retention: !duration 7d",
    Loader=DurationWithDaysLoader,
)

print(retention)

assert retention["retention"] == 604800.0


# Problem 13 — Better Validation Errors with Paths

A message such as:

```text
port must be an integer
```

is useful for a tiny document.

In a large document with many services, it is not enough.

A better message identifies the path:

```text
services[2].port must be an integer
```


### Step 1 — Validate one service with a path prefix


In [ ]:
def validate_service_at_path(raw: Any, path: str) -> dict:
    if not isinstance(raw, dict):
        raise ValueError(f"{path} must be a mapping")

    name = raw.get("name")
    port = raw.get("port")
    workers = raw.get("workers", 1)

    if not isinstance(name, str) or not name.strip():
        raise ValueError(f"{path}.name must be a non-empty string")

    if isinstance(port, bool) or not isinstance(port, int):
        raise ValueError(f"{path}.port must be an integer")

    if not 1 <= port <= 65535:
        raise ValueError(f"{path}.port must be between 1 and 65535")

    if isinstance(workers, bool) or not isinstance(workers, int) or workers <= 0:
        raise ValueError(f"{path}.workers must be a positive integer")

    return {
        "name": name.strip(),
        "port": port,
        "workers": workers,
    }


### Step 2 — Validate a sequence of services


In [ ]:
def validate_service_list(raw: Any) -> list[dict]:
    if not isinstance(raw, dict):
        raise ValueError("root must be a mapping")

    services = raw.get("services")

    if not isinstance(services, list):
        raise ValueError("services must be a list")

    result = []

    for index, service in enumerate(services):
        result.append(
            validate_service_at_path(
                service,
                f"services[{index}]",
            )
        )

    return result


### Step 3 — Observe the improved failure message


In [ ]:
bad_services_yaml = '''
services:
  - name: api
    port: 8000

  - name: worker
    port: 9000

  - name: broken
    port: not-a-number
'''

try:
    validate_service_list(
        yaml.safe_load(bad_services_yaml)
    )
except ValueError as exc:
    print(exc)


## Your Problem

Extend `validate_service_list` so that service names must also be unique.

Report the path of the later duplicate.


### Solution


In [ ]:
def validate_unique_service_list(raw: Any) -> list[dict]:
    if not isinstance(raw, dict):
        raise ValueError("root must be a mapping")

    services = raw.get("services")

    if not isinstance(services, list):
        raise ValueError("services must be a list")

    result = []
    seen_names = set()

    for index, service in enumerate(services):
        path = f"services[{index}]"

        normalized = validate_service_at_path(
            service,
            path,
        )

        name = normalized["name"]

        if name in seen_names:
            raise ValueError(
                f"{path}.name duplicates service name {name!r}"
            )

        seen_names.add(name)
        result.append(normalized)

    return result


duplicate_services_yaml = '''
services:
  - name: api
    port: 8000

  - name: worker
    port: 9000

  - name: api
    port: 8100
'''

try:
    validate_unique_service_list(
        yaml.safe_load(duplicate_services_yaml)
    )
except ValueError as exc:
    print(exc)


# Problem 14 — Redacting Secrets Before Logging

Configuration often contains secrets.

Printing the entire parsed dictionary can accidentally expose credentials in logs or notebooks.

We can create a sanitized copy before logging.


### Step 1 — Define sensitive key fragments


In [ ]:
SENSITIVE_FRAGMENTS = {
    "password",
    "secret",
    "token",
    "api_key",
    "private_key",
}


### Step 2 — Detect sensitive key names


In [ ]:
def is_sensitive_key(key: Any) -> bool:
    normalized = str(key).lower()

    return any(
        fragment in normalized
        for fragment in SENSITIVE_FRAGMENTS
    )


### Step 3 — Redact recursively

We create a new structure instead of mutating the runtime configuration.


In [ ]:
def redact_secrets(value: Any) -> Any:
    if isinstance(value, dict):
        result = {}

        for key, item in value.items():
            if is_sensitive_key(key):
                result[key] = "***REDACTED***"
            else:
                result[key] = redact_secrets(item)

        return result

    if isinstance(value, list):
        return [
            redact_secrets(item)
            for item in value
        ]

    return value


### Step 4 — Try it on nested configuration


In [ ]:
secret_yaml = '''
database:
  username: app
  password: super-secret

integrations:
  github:
    api_token: ghp_example
  payments:
    credentials:
      private_key: key-data
'''

runtime_secret_config = yaml.safe_load(secret_yaml)
safe_log_config = redact_secrets(runtime_secret_config)

print("Runtime value:")
pprint(runtime_secret_config)

print()
print("Safe logging value:")
pprint(safe_log_config)


## Your Problem

Add a list of service accounts, each containing a `name` and `token`.

Verify that tokens are redacted even when sensitive dictionaries are nested inside a list.


### Solution


In [ ]:
accounts_yaml = '''
accounts:
  - name: reporting
    token: report-token
  - name: billing
    token: billing-token
'''

accounts = yaml.safe_load(accounts_yaml)
redacted_accounts = redact_secrets(accounts)

pprint(redacted_accounts)

assert redacted_accounts["accounts"][0]["token"] == "***REDACTED***"
assert redacted_accounts["accounts"][1]["token"] == "***REDACTED***"


# Problem 15 — Versioned Configuration and Migration

Configuration formats change over time.

Suppose version 1 used:

```yaml
version: 1
server:
  host: localhost
  port: 8000
```

Version 2 uses:

```yaml
version: 2
network:
  bind_host: localhost
  bind_port: 8000
```

Instead of forcing all old files to change immediately, applications can migrate old structures into the newest internal representation.


### Step 1 — Write a migration from version 1 to version 2


In [ ]:
def migrate_v1_to_v2(config: dict) -> dict:
    server = config.get("server")

    if not isinstance(server, dict):
        raise ValueError("version 1 configuration requires 'server'")

    migrated = deepcopy(config)

    migrated["version"] = 2
    migrated["network"] = {
        "bind_host": server.get("host", "localhost"),
        "bind_port": server.get("port", 8000),
    }

    migrated.pop("server", None)

    return migrated


### Step 2 — Create a generic migration entry point


In [ ]:
def migrate_config(config: dict) -> dict:
    if not isinstance(config, dict):
        raise ValueError("configuration must be a mapping")

    version = config.get("version", 1)

    if version == 1:
        config = migrate_v1_to_v2(config)
        version = 2

    if version != 2:
        raise ValueError(f"unsupported configuration version: {version}")

    return config


### Step 3 — Migrate an old document


In [ ]:
old_config_yaml = '''
version: 1
server:
  host: 0.0.0.0
  port: 8080
logging:
  level: INFO
'''

old_config = yaml.safe_load(old_config_yaml)
new_config = migrate_config(old_config)

pprint(new_config)

assert new_config["version"] == 2
assert new_config["network"]["bind_host"] == "0.0.0.0"
assert new_config["network"]["bind_port"] == 8080
assert "server" not in new_config


## Your Problem

Add a version 2 validation function that requires:

- `network.bind_host` to be a non-empty string,
- `network.bind_port` to be a valid integer port.

Then migrate and validate the old configuration in one pipeline.


### Solution


In [ ]:
def validate_v2_config(config: dict) -> dict:
    if config.get("version") != 2:
        raise ValueError("configuration must be version 2")

    network = config.get("network")

    if not isinstance(network, dict):
        raise ValueError("'network' must be a mapping")

    host = network.get("bind_host")
    port = network.get("bind_port")

    if not isinstance(host, str) or not host.strip():
        raise ValueError("'network.bind_host' must be a non-empty string")

    if isinstance(port, bool) or not isinstance(port, int):
        raise ValueError("'network.bind_port' must be an integer")

    if not 1 <= port <= 65535:
        raise ValueError("'network.bind_port' must be between 1 and 65535")

    return config


migrated_and_validated = validate_v2_config(
    migrate_config(
        yaml.safe_load(old_config_yaml)
    )
)

pprint(migrated_and_validated)


# Problem 16 — Final Capstone: Build a Tutorial-Style Configuration Loader

We now have several independent pieces:

- safe YAML parsing,
- duplicate-key detection,
- environment expansion,
- configuration layering,
- validation,
- migration,
- secret redaction.

The final problem combines them into one application-oriented workflow.


## Scenario

An application has:

1. built-in defaults,
2. a user YAML document,
3. environment variables,
4. version migration,
5. application validation.

We want the final runtime configuration to be predictable and safe.


### Step 1 — Built-in defaults


In [ ]:
DEFAULT_CONFIG = {
    "version": 2,
    "network": {
        "bind_host": "127.0.0.1",
        "bind_port": 8000,
    },
    "logging": {
        "level": "INFO",
        "json": False,
    },
    "database": {
        "host": "localhost",
        "password": "",
    },
}


### Step 2 — User YAML

The file may contain environment placeholders.

It may also still be version 1.


In [ ]:
user_yaml = '''
version: 1

server:
  host: "${BIND_HOST:-0.0.0.0}"
  port: 8088

logging:
  level: WARNING
  json: true

database:
  host: "${DB_HOST:-localhost}"
  password: "${DB_PASSWORD}"
'''


### Step 3 — Parse with duplicate-key detection

This stage only answers:

> Is the YAML syntactically valid and free from duplicate mapping keys?


In [ ]:
parsed_user_config = yaml.load(
    user_yaml,
    Loader=DuplicateCheckingLoader,
)

pprint(parsed_user_config)


### Step 4 — Expand environment variables

We do this before migration because old-version fields may contain placeholders too.


In [ ]:
expanded_user_config = interpolate_environment(
    parsed_user_config,
    {
        "BIND_HOST": "0.0.0.0",
        "DB_HOST": "db.prod.internal",
        "DB_PASSWORD": "do-not-print-this",
    },
)

pprint(expanded_user_config)


### Step 5 — Migrate the document

The user configuration becomes version 2.


In [ ]:
migrated_user_config = migrate_config(
    expanded_user_config
)

pprint(migrated_user_config)


### Step 6 — Merge with defaults

The user document should override defaults, while unspecified values remain available.


In [ ]:
final_runtime_config = deep_merge(
    DEFAULT_CONFIG,
    migrated_user_config,
)

pprint(final_runtime_config)


### Step 7 — Validate the final version 2 structure


In [ ]:
validated_final_config = validate_v2_config(
    final_runtime_config
)

print("Final configuration is valid.")


### Step 8 — Create a safe logging view

We should not display the real database password.


In [ ]:
safe_final_view = redact_secrets(
    validated_final_config
)

pprint(safe_final_view)


### Step 9 — Serialize a redacted diagnostic snapshot

A diagnostic export should contain useful structure without exposing secrets.


In [ ]:
diagnostic_yaml = yaml.safe_dump(
    safe_final_view,
    sort_keys=False,
    allow_unicode=True,
)

print(diagnostic_yaml)


## Final Challenge

Wrap the entire pipeline in one function called:

```python
load_application_config(text, env)
```

The function should:

1. parse with duplicate-key detection,
2. expand environment placeholders,
3. migrate the configuration,
4. merge it with defaults,
5. validate version 2,
6. return the final runtime dictionary.

Then test it with the `user_yaml` document.


### Final Solution


In [ ]:
def load_application_config(
    text: str,
    env: dict[str, str],
) -> dict:
    try:
        parsed = yaml.load(
            text,
            Loader=DuplicateCheckingLoader,
        )
    except yaml.YAMLError as exc:
        raise ValueError(f"invalid YAML: {exc}") from exc

    if not isinstance(parsed, dict):
        raise ValueError("top-level YAML document must be a mapping")

    expanded = interpolate_environment(
        parsed,
        env,
    )

    migrated = migrate_config(
        expanded
    )

    merged = deep_merge(
        DEFAULT_CONFIG,
        migrated,
    )

    validated = validate_v2_config(
        merged
    )

    return validated


### Test the complete loader


In [ ]:
loaded_application_config = load_application_config(
    user_yaml,
    {
        "BIND_HOST": "0.0.0.0",
        "DB_HOST": "db.prod.internal",
        "DB_PASSWORD": "very-secret",
    },
)

pprint(
    redact_secrets(
        loaded_application_config
    )
)

assert loaded_application_config["version"] == 2
assert loaded_application_config["network"]["bind_host"] == "0.0.0.0"
assert loaded_application_config["network"]["bind_port"] == 8088
assert loaded_application_config["logging"]["level"] == "WARNING"
assert loaded_application_config["database"]["host"] == "db.prod.internal"


# Further Advanced Exercises

The main tutorial is complete.

Here are additional problems you can solve by extending the same ideas.

### Exercise A — Port collision detection

Add multiple network listeners and reject duplicate ports.

### Exercise B — Typed environment overrides

Support environment variables such as:

```text
APP__LOGGING__JSON=true
APP__NETWORK__BIND_PORT=9000
```

Parse their values into native YAML types.

### Exercise C — Configuration deprecation warnings

Suppose `logging.json` is replaced by `logging.format`.

Accept the old field temporarily but emit a warning.

### Exercise D — Custom `!bytes` tag

Support:

```yaml
memory_limit: !bytes 512MiB
```

Convert it into an integer number of bytes.

### Exercise E — Structural configuration diff

Given two configuration dictionaries, report:

- added paths,
- removed paths,
- changed values.

### Exercise F — Safe file loading

Create a function that:

- reads UTF-8,
- catches `OSError`,
- catches YAML parse errors,
- uses duplicate-key detection.

### Exercise G — Atomic configuration writes

Serialize to a temporary file first, then replace the destination.

### Exercise H — Cross-field validation

Reject a configuration when:

```yaml
environment: production
debug: true
```

### Exercise I — Multi-document resource validation

Use `safe_load_all` to parse several resources and reject duplicate `(kind, name)` pairs.

### Exercise J — Schema evolution

Add configuration version 3 and implement:

```text
v1 -> v2 -> v3
```

as sequential migration functions.


# Best Practices Recap

The examples in this notebook lead to a few general principles.

### Use safe parsing by default

For ordinary configuration:

```python
yaml.safe_load(text)
```

or a loader derived from `yaml.SafeLoader` is the normal choice.

### Do not confuse parsing with validation

A valid YAML document may still violate every rule your application expects.

### Convert to plain data before serialization

Dataclasses and domain objects can be transformed to dictionaries first.

This keeps serialization explicit and avoids Python-specific YAML tags.

### Reject ambiguity when configuration mistakes are expensive

Useful checks may include:

- duplicate keys,
- unknown keys,
- invalid ranges,
- duplicate identifiers,
- missing required values,
- cross-field conflicts.

### Treat environment variables as input

They still need validation.

### Never log secrets directly

Create a redacted copy for diagnostics.

### Prefer small, composable stages

A maintainable pipeline often looks like:

```text
text
  -> safe parse
  -> normalization
  -> environment expansion
  -> migration
  -> merge
  -> validation
  -> domain objects
```


# Closing Notes

YAML itself is only a data representation format.

The difficult part in real systems is usually everything around it:

- deciding which types are allowed,
- validating meaning,
- handling configuration evolution,
- composing multiple sources,
- reporting useful errors,
- keeping secrets safe.

That is why robust YAML code is usually less about a single `yaml.load(...)` call and more about designing a clear configuration pipeline.
